# Parte 3: Lógica de Predicados

En esta parte trabajarás con **lógica de predicados** usando cláusulas de Horn.

Los motores de **forward chaining** y **backward chaining** ya están dados — están en `src/forward_chaining.py` y `src/backward_chaining.py`.  
Tu tarea es **leer y entender los casos criminales** e implementar los hechos y reglas de cada caso en `crimes/`.

1. **Forward chaining:** Deriva todos los hechos posibles a partir de la evidencia.
2. **Backward chaining:** Prueba una meta específica encadenando reglas hacia atrás.
3. **Cuantificadores:** `∃` (exists) y `∀` (forall) para preguntas sobre el dominio.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.predicate_logic import KnowledgeBase, Predicate, Rule, Term, ExistsGoal, ForallGoal
from src.forward_chaining import forward_chain
from src.backward_chaining import backward_chain

## Estructura básica

Antes de trabajar con los casos criminales, entiende los componentes básicos.

**Convención de variables:**
- Las **variables** empiezan con `$`: `$X`, `$Y`, `$Z`
- Las **constantes** NO llevan `$`: `reynaldo`, `frasco_arsenico`, `cocina`

**Tipos:**
- `Term('reynaldo')` — constante
- `Term('$X')` — variable (unificable)
- `Predicate('culpable', (Term('reynaldo'),))` — predicado ground
- `Predicate('culpable', (Term('$X'),))` — predicado con variable
- `Rule(head=..., body=(...))` — regla: head :- body1, body2, ...

In [ ]:
# Términos
reynaldo = Term('reynaldo')   # Constante
X = Term('$X')                # Variable

print(f"'reynaldo' es variable: {reynaldo.is_variable}")
print(f"'$X' es variable: {X.is_variable}")

# Predicados
p1 = Predicate('culpable', (reynaldo,))
p2 = Predicate('culpable', (X,))
print(f"\nPredicado ground:    {p1}")
print(f"Predicado con var:   {p2}")

# Regla: culpable($X) :- evidencia_directa($X), sin_coartada($X)
regla = Rule(
    head=Predicate('culpable', (Term('$X'),)),
    body=(
        Predicate('evidencia_directa', (Term('$X'),)),
        Predicate('sin_coartada',      (Term('$X'),)),
    )
)
print(f"\nRegla: {regla}")

In [ ]:
# Ejemplo mínimo: KB pequeña con forward chaining
kb = KnowledgeBase()

alice = Term('alice')
pistola = Term('pistola')

kb.add_fact(Predicate('huellas_en',      (alice, pistola)))
kb.add_fact(Predicate('arma_del_crimen', (pistola,)))
kb.add_fact(Predicate('sin_coartada',    (alice,)))

kb.add_rule(Rule(
    head=Predicate('evidencia_directa', (Term('$X'),)),
    body=(
        Predicate('huellas_en',      (Term('$X'), Term('$A'))),
        Predicate('arma_del_crimen', (Term('$A'),)),
    )
))
kb.add_rule(Rule(
    head=Predicate('culpable', (Term('$X'),)),
    body=(
        Predicate('evidencia_directa', (Term('$X'),)),
        Predicate('sin_coartada',      (Term('$X'),)),
    )
))

result = forward_chain(kb)
base = {f.predicate for f in kb.facts}
nuevos = sorted(result.derived_facts - base, key=repr)
print("Hechos derivados:")
for fact in nuevos:
    print(f"  {fact}")

## Ejercicio 1: Los cinco casos criminales

Abre los archivos en `crimes/`. Cada uno tiene:
- Un **docstring** en dos párrafos: el primero enuncia los hechos (uno por línea = un `add_fact`), el segundo las reglas de deducción (una por línea = un `add_rule`).
- Una sección `# === YOUR CODE HERE ===` donde debes escribir los `add_fact` y `add_rule`.
- Una sección `CASE` con las consultas que deben verificarse.

**Archivos a implementar:**
1. `crimes/veneno_villa_espinas.py` — Asesinato con arsénico
2. `crimes/robo_expreso_sur.py` — Robo de joyas en tren nocturno
3. `crimes/sabotaje_pharmax.py` — Sabotaje a laboratorio farmacéutico
4. `crimes/herencia_hacienda_rosal.py` — Envenenamiento por herencia
5. `crimes/red_puerto_sombras.py` — Red de contrabando portuario

Para cada caso, ejecuta las consultas con backward chaining y verifica que todas sean correctas.

In [ ]:
# Verificar todos los casos implementados
import importlib

casos = [
    'crimes.veneno_villa_espinas',
    'crimes.robo_expreso_sur',
    'crimes.sabotaje_pharmax',
    'crimes.herencia_hacienda_rosal',
    'crimes.red_puerto_sombras',
]

total = 0
correctas = 0

for mod_name in casos:
    caso = importlib.import_module(mod_name).CASE
    kb = caso.create_kb()
    print(f"\n{'='*50}")
    print(f"  {caso.title}")
    print(f"  Sospechosos: {', '.join(caso.suspects)}")
    print(f"{'='*50}")
    for q in caso.queries:
        r = backward_chain(kb, q.goal)
        icon = '✓' if r.success else '✗'
        print(f"  [{icon}] {q.description}")
        total += 1
        if r.success:
            correctas += 1

print(f"\n{'='*50}")
print(f"  Total: {correctas}/{total} consultas respondidas")
print(f"  {'✓ Todos los casos correctos!' if correctas == total else '✗ Hay consultas sin respuesta — revisa tu implementación'}")

## Ejercicio 2: Forward Chaining — derivar todos los hechos

El forward chaining parte de los hechos base y aplica reglas hasta que no se pueden derivar más hechos nuevos (punto fijo).

**Observa:** qué hechos se derivan, en qué orden, y cuántas iteraciones necesita.

In [ ]:
from crimes.veneno_villa_espinas import CASE

kb = CASE.create_kb()
result = forward_chain(kb)

print(f"Iteraciones necesarias: {result.iterations}")
print("\nPasos de derivación:")
for step in result.steps:
    print(f"  [{step.iteration}] {step.description}")

print("\nHechos derivados (no estaban en la KB base):")
base = {f.predicate for f in kb.facts}
nuevos = sorted(result.derived_facts - base, key=repr)
for fact in nuevos:
    print(f"  {fact}")

In [ ]:
# Comparar forward chaining en los 5 casos
for mod_name in casos:
    caso = importlib.import_module(mod_name).CASE
    kb = caso.create_kb()
    result = forward_chain(kb)
    base = {f.predicate for f in kb.facts}
    culpables = [f for f in result.derived_facts if f.name == 'culpable']
    descartados = [f for f in result.derived_facts if f.name == 'descartado']
    print(f"{caso.title}")
    print(f"  Iteraciones: {result.iterations}  |  Culpables: {culpables}  |  Descartados: {descartados}")

## Ejercicio 3: Backward Chaining — árbol de prueba

El backward chaining parte de una meta y encadena reglas hacia atrás hasta encontrar hechos base.

**Diferencias con forward chaining:**
- Es **orientado a metas**: solo prueba lo necesario.
- Construye un **árbol de prueba** legible.
- Responde preguntas sobre **individuos específicos**.

In [ ]:
from crimes.veneno_villa_espinas import CASE
kb = CASE.create_kb()

for nombre in ['reynaldo', 'margot', 'pablo', 'bernardo']:
    result = backward_chain(kb, Predicate('culpable', (Term(nombre),)))
    print(f"\n--- ¿{nombre.capitalize()} es culpable? {'SÍ' if result.success else 'NO'} ---")
    for step in result.proof_steps:
        print(f"  {step}")

In [ ]:
# Probar todas las consultas del caso con árbol de prueba detallado
from crimes.red_puerto_sombras import CASE as caso_puerto
kb_puerto = caso_puerto.create_kb()

for q in caso_puerto.queries:
    r = backward_chain(kb_puerto, q.goal)
    icon = '✓' if r.success else '✗'
    print(f"[{icon}] {q.description}")
    # Descomentar para ver el árbol de prueba completo:
    # for step in r.proof_steps:
    #     print(f"    {step}")

### Preguntas para reflexionar:
- ¿Por qué el backward chaining necesita menos pasos que el forward para probar una meta específica?
- ¿Qué ocurre cuando intentas probar algo falso? ¿Cuántos pasos se dan antes de fallar?
- ¿Cómo detecta el sistema ciclos infinitos?

## Ejercicio 4: Cuantificadores — ∃ (Exists) y ∀ (Forall)

Los cuantificadores permiten preguntas sobre **todos** o **algún** elemento del dominio:

- `ExistsGoal('$X', Predicate('culpable', (Term('$X'),)))` → ¿∃X tal que culpable(X)?
- `ForallGoal('$X', dominio_pred, propiedad_pred)` → ¿∀X en dominio, propiedad(X)?

**Nota:** Los cuantificadores se usan solo en las **consultas** (queries), no en el cuerpo de las reglas.

In [ ]:
from crimes.veneno_villa_espinas import CASE
kb = CASE.create_kb()

# ExistsGoal: ∃X. culpable(X) — ¿hay algún culpable?
exists_q = ExistsGoal('$X', Predicate('culpable', (Term('$X'),)))
r = backward_chain(kb, exists_q)
print(f"∃X. culpable(X) → {'Sí' if r.success else 'No'}")
# Las sustituciones contienen quién es el culpable
for subst in r.substitutions:
    print(f"  Sustitución: {subst}")

In [ ]:
# ExistsGoal: ∃X. descartado(X) — ¿hay algún descartado?
exists_desc = ExistsGoal('$X', Predicate('descartado', (Term('$X'),)))
r = backward_chain(kb, exists_desc)
print(f"∃X. descartado(X) → {'Sí' if r.success else 'No'}")
print(f"Sustituciones: {r.substitutions}")

# ForallGoal: ∀X. descartado(X) → sin_coartada no aplica — todos los descartados tienen coartada
# Aquí usamos coartada_verificada como dominio
forall_q = ForallGoal(
    '$X',
    Predicate('descartado', (Term('$X'),)),
    Predicate('lejos_de_escena', (Term('$X'),)),
)
r2 = backward_chain(kb, forall_q)
print(f"\n∀X. descartado(X) → lejos_de_escena(X): {'Sí' if r2.success else 'No'}")

In [ ]:
from crimes.sabotaje_pharmax import CASE as caso_pharmax
from crimes.red_puerto_sombras import CASE as caso_puerto

# ∀X. culpable(X) → acceso_en_momento(X)
# ¿Todo culpable tuvo acceso en el momento del sabotaje?
kb_pharmax = caso_pharmax.create_kb()
forall_culpable = ForallGoal(
    '$X',
    Predicate('culpable', (Term('$X'),)),
    Predicate('acceso_en_momento', (Term('$X'),)),
)
r = backward_chain(kb_pharmax, forall_culpable)
print(f"∀X. culpable(X) → acceso_en_momento(X): {'Sí' if r.success else 'No'}")

# ∃R. red_activa(R) en el caso del puerto
kb_puerto = caso_puerto.create_kb()
exists_red = ExistsGoal('$R', Predicate('red_activa', (Term('$R'),)))
r2 = backward_chain(kb_puerto, exists_red)
print(f"∃R. red_activa(R): {'Sí' if r2.success else 'No'}")